# City action reduction-potential — calculation sheet (WORKING)

Runs IPCC option potentials *backward* onto a city's inventory, and adds a **cost-effectiveness axis** from the SPM.7 cost bins. All inputs live in CSVs under `releases/2023/data/calc_inputs/`; edit the CSVs, not the code.

| file | bucket |
|---|---|
| `ipcc_parameters.csv` | IPCC values (EFs, intensities, abatement %) with source + confidence |
| `calc_options.csv` | option register (computed + excluded), segment, tier, calc_type, params, `ipcc_option` for cost lookup |
| `city_inventory_santiago.csv` | INVENTORY baselines per GPC segment |
| `city_parameters_santiago.csv` | CITY knobs (grid EF, blend, mode-shift) |

`segment` = city inventory baseline (not the global potential). **Cost is global/option-level (SPM.7 bins), not city-adjusted** — flagged below.

In [ ]:
import pandas as pd
from pathlib import Path
D = Path('releases/2023/data/calc_inputs')
ipcc  = pd.read_csv(D/'ipcc_parameters.csv').set_index('parameter_id')
opts  = pd.read_csv(D/'calc_options.csv')
inv   = pd.read_csv(D/'city_inventory_santiago.csv').set_index('segment')['baseline_mtco2e']
cityp = pd.read_csv(D/'city_parameters_santiago.csv').set_index('parameter')['value']
P = ipcc['value'].to_dict()
grid_ef=float(cityp['grid_ef']); blend=float(cityp['blend_ethanol_share']); mode_shift=float(cityp['mode_shift_fraction'])

## Compute — quantity (reduction) and cost-effectiveness

In [ ]:
def abatement(r):
    if r.calc_type=='none': return float('nan')
    if r.calc_type=='identity':
        return 1 - (P[r.intensity_to_id]*grid_ef)/(P[r.intensity_from_id]*P[r.ef_from_id]*(1-blend))
    if r.calc_type=='assessed':  return P[r.abatement_id]
    if r.calc_type=='modeshift': return mode_shift*(1-1/P[r.abatement_id])
    raise ValueError(r.calc_type)

# --- cost-effectiveness from global SPM.7 cost bins (option-level, NOT city-adjusted) ---
g = pd.read_csv('releases/2023/data/spm7a_wgiii_options_2030_clean.csv').set_index('option')
MID = {'cost_lt0':-25,'cost_0_20':10,'cost_20_50':35,'cost_50_100':75,'cost_100_200':150}  # USD/tCO2e bin midpoints
def cost_metrics(ipcc_opt):
    if pd.isna(ipcc_opt) or not str(ipcc_opt): return (float('nan'),float('nan'))
    mem=[m.strip() for m in str(ipcc_opt).split('|') if m.strip() in g.index]
    if not mem: return (float('nan'),float('nan'))
    rows=g.loc[mem]
    priced=sum(rows[b].fillna(0).sum() for b in MID); weighted=sum(rows[b].fillna(0).sum()*m for b,m in MID.items())
    tot=rows['total'].fillna(0).sum(); unalloc=rows['cost_unallocated'].fillna(0).sum()
    ci = round(weighted/priced,1) if priced>0 else float('nan')
    return (ci, round(unalloc/tot,2) if tot else float('nan'))

df = opts.copy()
df['abatement'] = df.apply(abatement, axis=1).round(3)
df['baseline']  = df['gpc_segment'].map(inv)
df['reduction'] = (df['baseline']*df['abatement']*df['uptake']).round(2)
df[['cost_usd_tco2e','share_unpriced']] = df['ipcc_option'].apply(lambda x: pd.Series(cost_metrics(x)))
computed = df[df.calc_type!='none'].copy()
computed.sort_values('reduction',ascending=False)[['tier','option','baseline','abatement','reduction','cost_usd_tco2e','share_unpriced']]

## Two rankings — quantity vs cost-effectiveness
**Quantity** = biggest reducers first (envelope per segment). **Cost** = cheapest $/tCO2e first (global cost bins). They answer different questions; compare the orderings.

In [ ]:
env = computed.loc[computed.groupby('gpc_segment')['reduction'].idxmax()].copy()

by_quantity = env.sort_values('reduction', ascending=False)[['option','reduction','cost_usd_tco2e']]
by_cost     = env.sort_values('cost_usd_tco2e')[['option','cost_usd_tco2e','reduction','share_unpriced']]
print('=== RANK BY QUANTITY (MtCO2e, biggest first) ===');  print(by_quantity.to_string(index=False))
print('\n=== RANK BY COST-EFFECTIVENESS (USD/tCO2e, cheapest first; global, not city) ==='); print(by_cost.to_string(index=False))

## 2-D screen — the MACC-style view
Priority quadrant = **high reduction + low/negative cost**. `share_unpriced` flags options whose cost IPCC left unallocated (e.g. EVs) — treat their cost as unknown, not zero.

In [ ]:
scr = env.copy()
scr['cost_tier'] = pd.cut(scr['cost_usd_tco2e'], [-100,0,50,200], labels=['cost-saving','low','high'])
scr['size_tier'] = pd.cut(scr['reduction'], [0,0.25,0.6,99], labels=['small','medium','large'])
scr.sort_values(['cost_usd_tco2e','reduction'],ascending=[True,False])[['option','reduction','size_tier','cost_usd_tco2e','cost_tier','share_unpriced']]

## Full register — every option accounted for

In [ ]:
df[['tier','option','gpc_segment','calc_type','abatement','reduction','cost_usd_tco2e','status']]

## Notes / gaps
- **Cost is global** (SPM.7 option-level bins, 2015-20 vintage), applied via `ipcc_option`. It is NOT city-adjusted — local capital/energy prices would shift it. A city-specific MACC is a separate (harder) sourcing task.
- `share_unpriced` > 0: IPCC left that option's cost unallocated (EVs, demand-side) — cost shown is from priced bins only; don't read as cheap.
- Composite rows (Buildings SER, industry, process) average cost across their member IPCC options (`|`-joined in `ipcc_option`).
- All the usual placeholders apply (illustrative intensities/rates, waste MACCs, 2050 buildings horizon); excluded rows (grid supply, AFOLU, shipping/aviation, F-gas, embodied) are not city actions — see `status`.